<a href="https://colab.research.google.com/github/susara20010420/Workplace-Safety-Insights-from-the-Industrial-Safety-Health-Analytics-Dataset/blob/main/v3_Bayesian_Methodology_Companion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bayesian Severity Companion Model
Methodology-aligned template.

In [1]:
# Install
!pip -q install pymc arviz sentence-transformers scikit-learn

In [2]:
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import json
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

from google.colab import drive
drive.mount('/content/drive')
import os

project_folder="/content/drive/MyDrive/Work Place Safety Insights"

train_df=pd.read_csv(f"{project_folder}/train.csv")
test_df=pd.read_csv(f"{project_folder}/test.csv")
transformer_predictions=pd.read_csv(f"{project_folder}/transformer_predictions.csv")

with open(f"{project_folder}/class_weights.json") as f:
    class_weights=json.load(f)


Mounted at /content/drive


## MiniLM Embeddings + PCA

In [3]:
encoder=SentenceTransformer("all-MiniLM-L6-v2")

train_emb=encoder.encode(train_df["description"].tolist(),show_progress_bar=True)
test_emb=encoder.encode(test_df["description"].tolist(),show_progress_bar=True)

# Reduced PCA components from 100 to 20 as suggested to address overfitting with small dataset
pca=PCA(n_components=20,random_state=42)
X_train=pca.fit_transform(train_emb)
X_test=pca.transform(test_emb)

y_train=train_df["severity_label"].values
y_test=test_df["severity_label"].values

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

## Hierarchical indices

In [4]:
train_df["sector_idx"],sector_names=pd.factorize(train_df["Industry Sector"])
test_df["sector_idx"]=pd.Categorical(test_df["Industry Sector"],categories=sector_names).codes

train_df["plant_idx"],plant_names=pd.factorize(train_df["Local"])
test_df["plant_idx"]=pd.Categorical(test_df["Local"],categories=plant_names).codes


## Hierarchical Ordinal Bayesian Model

In [5]:
from pymc.distributions.transforms import ordered

n_features=X_train.shape[1] # This will now be 20 due to PCA change
n_classes=len(np.unique(y_train))
n_sectors=len(sector_names) # Kept for potential future use or consistency
n_plants=len(plant_names)

with pm.Model() as ordinal_model:
    # Non-centered beta with better priors (HalfNormal(0.5) instead of HalfCauchy(1))
    tau = pm.HalfNormal("tau", 0.5)
    lam = pm.HalfNormal("lam", 0.5, shape=n_features)
    beta_raw = pm.Normal("beta_raw", 0, 1, shape=n_features)
    beta = pm.Deterministic("beta", beta_raw * tau * lam)

    # Hierarchical effects (simplified to plant_effect only with tighter prior)
    plant_sd = pm.HalfNormal("plant_sd", 0.5)
    plant_offset = pm.Normal("plant_offset", 0, 1, shape=n_plants)
    plant_effect = pm.Deterministic("plant_effect", plant_offset * plant_sd)

    # Linear predictor without sector_effect for simplification
    eta = pm.math.dot(X_train, beta) + plant_effect[train_df["plant_idx"].values]

    # Reverting to explicit ordered cutpoints using cumulative sums for robust initialization
    # The 'ordered' transform was causing initialization issues.
    cutpoints_0 = pm.Normal("cutpoints_0", mu=0, sigma=2)
    # Using HalfNormal(0.5) for a tighter prior on differences
    cutpoint_diffs = pm.HalfNormal("cutpoint_diffs", sigma=0.5, shape=n_classes - 2)
    cutpoints = pm.Deterministic("cutpoints",
        pm.math.concatenate([[cutpoints_0], cutpoints_0 + pm.math.cumsum(cutpoint_diffs)]))

    severity = pm.OrderedLogistic(
        "severity",
        eta=eta,
        cutpoints=cutpoints,
        observed=y_train)

    # Increased target_accept, tune, max_treedepth and set cores=1 for better convergence
    trace = pm.sample(1000, tune=2000, target_accept=0.99, max_treedepth=15, return_inferencedata=True, cores=1)

    posterior_pred = pm.sample_posterior_predictive(trace)

Output()

Output()

## Posterior summaries

In [6]:
summary=az.summary(trace,hdi_prob=0.95)
summary.to_csv(f"{project_folder}/posterior_summary.csv")
display(summary.head())


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
beta[0],-3.927,0.292,-4.501,-3.400,0.006,0.006,2209.0,1439.0,1.0
beta[1],1.614,0.467,0.689,2.485,0.010,0.009,2332.0,1816.0,1.0
beta[2],4.803,0.499,3.819,5.740,0.010,0.009,2279.0,1900.0,1.0
beta[3],2.035,0.727,0.591,3.399,0.015,0.014,2282.0,1367.0,1.0
beta[4],-0.296,0.353,-1.057,0.278,0.008,0.006,1942.0,1844.0,1.0


## Posterior prediction on test set (replace with full predictive routine as needed)

In [7]:
# Expected utility / alert threshold example
# Compute posterior predictive probabilities for test observations
# (extend using pm.Data for production inference)

from google.colab import drive
drive.mount('/content/drive')
import os

project_folder = "/content/drive/MyDrive/Work Place Safety Insights"
os.makedirs(project_folder, exist_ok=True)

decision_results=transformer_predictions.copy()
decision_results["alert"]=decision_results["confidence"]>0.70

decision_results.to_csv(
    f"{project_folder}/decision_layer_results.csv",
    index=False)

transformer_predictions.to_csv(
    f"{project_folder}/bayesian_predictions.csv",
    index=False)

print("Outputs saved.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Outputs saved.
